In [ ]:
import argparse
from pathlib import Path
from dataclasses import dataclass

import numpy as np
import matplotlib.pyplot as plt

import sys
root = Path.cwd().parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from stepsic.data import CosmoData
from stepsic.field import cubic_voxels, fourier_grid

In [ ]:
import logging
log = logging.getLogger(__name__)
logging.basicConfig(level=logging.INFO)

In [ ]:
from typing import Literal
MassAssignment = Literal['ngp', 'cic', 'tsc']

In [ ]:
@dataclass(frozen=True)
class PowerSpectrum1D:
    k: np.ndarray          # [1/length]
    pk: np.ndarray         # [length^3]
    nmodes: np.ndarray     # counts per bin

In [ ]:
class MassAssignmentScheme:
    def __init__(self):
        super().__init__()

    def _assign_ngp(
        pos: np.ndarray,
        mass: np.ndarray,
        boxsize: np.ndarray,
        nmesh: tuple[int, int, int]
    ) -> np.ndarray:
        '''Nearest-grid-point deposition (periodic).'''
        nx, ny, nz = nmesh
        grid = np.zeros((nx, ny, nz), dtype=np.float64)

        # wrap into [0, L)
        x = np.mod(pos, boxsize)

        # index in [0, N-1]
        i = np.floor(x[:, 0] / boxsize[0] * nx).astype(np.int64) % nx
        j = np.floor(x[:, 1] / boxsize[1] * ny).astype(np.int64) % ny
        k = np.floor(x[:, 2] / boxsize[2] * nz).astype(np.int64) % nz

        np.add.at(grid, (i, j, k), mass)
        return grid

    def _assign_cic(
        pos: np.ndarray,
        mass: np.ndarray,
        boxsize: np.ndarray,
        nmesh: tuple[int, int, int]
    ) -> np.ndarray:
        '''Cloud-in-cell deposition (periodic).'''
        nx, ny, nz = nmesh
        grid = np.zeros((nx, ny, nz), dtype=np.float64)

        x = np.mod(pos, boxsize)
        gx = x[:, 0] / boxsize[0] * nx
        gy = x[:, 1] / boxsize[1] * ny
        gz = x[:, 2] / boxsize[2] * nz

        i0 = np.floor(gx).astype(np.int64) % nx
        j0 = np.floor(gy).astype(np.int64) % ny
        k0 = np.floor(gz).astype(np.int64) % nz

        tx = gx - np.floor(gx)
        ty = gy - np.floor(gy)
        tz = gz - np.floor(gz)

        i1 = (i0 + 1) % nx
        j1 = (j0 + 1) % ny
        k1 = (k0 + 1) % nz

        wx0, wx1 = 1.0 - tx, tx
        wy0, wy1 = 1.0 - ty, ty
        wz0, wz1 = 1.0 - tz, tz

        # 8 corners
        np.add.at(grid, (i0, j0, k0), mass * wx0 * wy0 * wz0)
        np.add.at(grid, (i0, j0, k1), mass * wx0 * wy0 * wz1)
        np.add.at(grid, (i0, j1, k0), mass * wx0 * wy1 * wz0)
        np.add.at(grid, (i0, j1, k1), mass * wx0 * wy1 * wz1)
        np.add.at(grid, (i1, j0, k0), mass * wx1 * wy0 * wz0)
        np.add.at(grid, (i1, j0, k1), mass * wx1 * wy0 * wz1)
        np.add.at(grid, (i1, j1, k0), mass * wx1 * wy1 * wz0)
        np.add.at(grid, (i1, j1, k1), mass * wx1 * wy1 * wz1)
        return grid

    def _assign_tsc(
        pos: np.ndarray,
        mass: np.ndarray,
        boxsize: np.ndarray,
        nmesh: tuple[int, int, int]
    ) -> np.ndarray:
        '''Triangular-shaped-cloud deposition (periodic).'''
        nx, ny, nz = nmesh
        grid = np.zeros((nx, ny, nz), dtype=np.float64)

        x = np.mod(pos, boxsize)
        gx = x[:, 0] / boxsize[0] * nx
        gy = x[:, 1] / boxsize[1] * ny
        gz = x[:, 2] / boxsize[2] * nz

        i1 = np.floor(gx).astype(np.int64) % nx
        j1 = np.floor(gy).astype(np.int64) % ny
        k1 = np.floor(gz).astype(np.int64) % nz

        dx = gx - np.floor(gx)
        dy = gy - np.floor(gy)
        dz = gz - np.floor(gz)

        i0 = (i1 - 1) % nx
        j0 = (j1 - 1) % ny
        k0 = (k1 - 1) % nz

        i2 = (i1 + 1) % nx
        j2 = (j1 + 1) % ny
        k2 = (k1 + 1) % nz

        wx0 = 0.5 * (1.5 - dx)**2
        wx1 = 0.75 - (dx - 1.0)**2
        wx2 = 0.5 * (dx - 0.5)**2

        wy0 = 0.5 * (1.5 - dy)**2
        wy1 = 0.75 - (dy - 1.0)**2
        wy2 = 0.5 * (dy - 0.5)**2

        wz0 = 0.5 * (1.5 - dz)**2
        wz1 = 0.75 - (dz - 1.0)**2
        wz2 = 0.5 * (dz - 0.5)**2

        # 27 corners
        np.add.at(grid, (i0, j0, k0), mass * wx0 * wy0 * wz0)
        np.add.at(grid, (i0, j0, k1), mass * wx0 * wy0 * wz1)
        np.add.at(grid, (i0, j0, k2), mass * wx0 * wy0 * wz2)
        np.add.at(grid, (i0, j1, k0), mass * wx0 * wy1 * wz0)
        np.add.at(grid, (i0, j1, k1), mass * wx0 * wy1 * wz1)
        np.add.at(grid, (i0, j1, k2), mass * wx0 * wy1 * wz2)
        np.add.at(grid, (i0, j2, k0), mass * wx0 * wy2 * wz0)
        np.add.at(grid, (i0, j2, k1), mass * wx0 * wy2 * wz1)
        np.add.at(grid, (i0, j2, k2), mass * wx0 * wy2 * wz2)
        np.add.at(grid, (i1, j0, k0), mass * wx1 * wy0 * wz0)
        np.add.at(grid, (i1, j0, k1), mass * wx1 * wy0 * wz1)
        np.add.at(grid, (i1, j0, k2), mass * wx1 * wy0 * wz2)
        np.add.at(grid, (i1, j1, k0), mass * wx1 * wy1 * wz0)
        np.add.at(grid, (i1, j1, k1), mass * wx1 * wy1 * wz1)
        np.add.at(grid, (i1, j1, k2), mass * wx1 * wy1 * wz2)
        np.add.at(grid, (i1, j2, k0), mass * wx1 * wy2 * wz0)
        np.add.at(grid, (i1, j2, k1), mass * wx1 * wy2 * wz1)
        np.add.at(grid, (i1, j2, k2), mass * wx1 * wy2 * wz2)
        np.add.at(grid, (i2, j0, k0), mass * wx2 * wy0 * wz0)
        np.add.at(grid, (i2, j0, k1), mass * wx2 * wy0 * wz1)
        np.add.at(grid, (i2, j0, k2), mass * wx2 * wy0 * wz2)
        np.add.at(grid, (i2, j1, k0), mass * wx2 * wy1 * wz0)
        np.add.at(grid, (i2, j1, k1), mass * wx2 * wy1 * wz1)
        np.add.at(grid, (i2, j1, k2), mass * wx2 * wy1 * wz2)
        np.add.at(grid, (i2, j2, k0), mass * wx2 * wy2 * wz0)
        np.add.at(grid, (i2, j2, k1), mass * wx2 * wy2 * wz1)
        np.add.at(grid, (i2, j2, k2), mass * wx2 * wy2 * wz2)
        return grid

In [ ]:
def estimate_pk_1d(
    *,
    pos: np.ndarray,
    boxsize: np.ndarray,
    nmesh: tuple[int, int, int],
    mass: np.ndarray | None = None,
    assignment: MassAssignment = 'tsc',
    kbins: np.ndarray | None = None,
) -> PowerSpectrum1D:
    """
    Estimate isotropically-averaged P(k) from particles in a rectangular periodic box.

    Parameters
    ----------
    pos
        Particle positions in the periodic box, shape (N, 3), same length units as boxsize.
    boxsize
        (Lx, Ly, Lz) box dimensions.
    nmesh
        (Nx, Ny, Nz) mesh resolution.
    mass
        Particle masses (optional). If None, assumes equal mass.
    assignment
        Mass assignment scheme: 'ngp', 'cic', or 'tsc'.
    kbins
        Bin edges in k [1/length]. If None, uses linear bins up to k_Nyquist(min axis).

    Returns
    -------
    PowerSpectrum1D
        Bin centers k, P(k), and mode counts.
    """
    pos = np.asarray(pos, dtype=np.float64)
    boxsize = np.asarray(boxsize, dtype=np.float64)
    if mass is None:
        mass = np.ones(pos.shape[0], dtype=np.float64)
    else:
        mass = np.asarray(mass, dtype=np.float64)

    nx, ny, nz = nmesh
    dx, dy, dz = boxsize[0] / nx, boxsize[1] / ny, boxsize[2] / nz
    vol = float(boxsize[0] * boxsize[1] * boxsize[2])
    dvol = float(dx * dy * dz)

    if assignment == "ngp":
        rho = _assign_ngp(pos, mass, boxsize, nmesh)
    elif assignment == "cic":
        rho = _assign_cic(pos, mass, boxsize, nmesh)
    else:
        raise ValueError(f"Unknown assignment={assignment!r}")

    rho_bar = rho.mean()
    if not np.isfinite(rho_bar) or rho_bar <= 0:
        raise ValueError("Non-positive mean density encountered.")

    delta = rho / rho_bar - 1.0
    delta -= delta.mean()  # enforce DC=0 robustly

    # FFT conventions:
    # numpy fft is unnormalized forward. Continuous FT approx: delta(k) ~ dV * FFT[delta(x)].
    dk_field = np.fft.rfftn(delta)
    delta_k = dvol * dk_field

    # power per mode: P(k) = < |delta(k)|^2 > / V
    p_mode = (np.abs(delta_k) ** 2) / vol

    _, kmod = fourier_grid(nmesh, dk, hermitian=True)

    # default bins: linear in k up to min Nyquist
    k_nyq = np.pi / min(dx, dy, dz)
    k_fund = 2.0 * np.pi / max(boxsize)  # conservative "lowest" fundamental
    if kbins is None:
        nb = max(32, int(np.sqrt(nx * ny * nz) // 4))
        kbins = np.linspace(0.0, k_nyq, nb + 1)

    # bin excluding k=0
    kvals = kmod.ravel()
    pvals = p_mode.ravel()
    mask = kvals > 0.0
    kvals = kvals[mask]
    pvals = pvals[mask]

    inds = np.digitize(kvals, kbins) - 1
    good = (inds >= 0) & (inds < (kbins.size - 1))
    inds = inds[good]
    kvals = kvals[good]
    pvals = pvals[good]

    nmodes = np.bincount(inds, minlength=kbins.size - 1).astype(np.int64)
    pk_sum = np.bincount(inds, weights=pvals, minlength=kbins.size - 1)

    pk = np.full(kbins.size - 1, np.nan, dtype=np.float64)
    nonzero = nmodes > 0
    pk[nonzero] = pk_sum[nonzero] / nmodes[nonzero]

    kcen = 0.5 * (kbins[:-1] + kbins[1:])
    return PowerSpectrum1D(k=kcen, pk=pk, nmodes=nmodes)


In [ ]:
def compute_pk_from_snapshot(
    snapshot: str | Path,
    *,
    nmesh: tuple[int, int, int],
    boxsize: tuple[float, float, float],
    part_type: int = 1,
    assignment: str = "cic",
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Compute the isotropically-averaged 1D power spectrum P(k)
    from a particle snapshot in a constant-resolution rectangular box.

    Parameters
    ----------
    snapshot
        Path to the snapshot file.
    nmesh
        Grid resolution (Nx, Ny, Nz).
    boxsize
        Box dimensions (Lx, Ly, Lz), same units as particle positions.
    part_type
        Particle type index (default: 1).
    assignment
        Mass-assignment scheme: 'ngp' or 'cic'.

    Returns
    -------
    k : ndarray
        Bin centers [1 / length].
    pk : ndarray
        Power spectrum P(k) [length^3].
    nmodes : ndarray
        Number of Fourier modes contributing to each bin.
    """
    snapshot = Path(snapshot)
    box = np.asarray(boxsize, dtype=np.float64)

    log.info(
        "Computing P(k): snapshot=%s, nmesh=%s, boxsize=%s, assignment=%s",
        snapshot,
        nmesh,
        boxsize,
        assignment,
    )

    data = CosmoData.load_snapshot(snapshot, part_type=part_type)

    pk = estimate_pk_1d(
        pos=data.pos,
        mass=data.mass,
        boxsize=box,
        nmesh=nmesh,
        assignment=assignment,
    )

    ok = np.isfinite(pk.pk) & (pk.nmodes > 0)

    return pk.k[ok], pk.pk[ok], pk.nmodes[ok]

In [ ]:
k, pk, nm = compute_pk_from_snapshot(
    snapshot=Path('../output/stepsic_Lx200_Ly200_Lz200_R3D100_D4D75_z63.hdf5'),
    nmesh=(128, 128, 128),
    boxsize=(200.0, 200.0, 200.0),
    part_type=1,
    assignment="cic",
)

np.savetxt(
    "../output/pk_1d.txt",
    np.column_stack([k, pk, nm]),
    header="k  Pk  Nmodes",
)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.loglog(k, pk, lw=2)
ax.set_xlabel(r"$k$ [$h\,\mathrm{Mpc}^{-1}$]")
ax.set_ylabel(r"$P(k)$ [$h^{-3}\,\mathrm{Mpc}^3$]")
ax.set_title("Isotropic Power Spectrum")
plt.tight_layout()
plt.show()